# Supply Chain Silver & Gold Layer Transformations

Builds the medallion architecture on top of the bronze layer:

**Silver layer** (`workspace.supply_chain_silver`): Cleaned, deduplicated, referentially validated tables with trimmed strings and `_ingested_at` audit column. Dimension tables are built first so fact tables can enforce foreign-key integrity.

**Gold layer** (`workspace.supply_chain_gold`): Business-ready aggregated tables for analytics and dashboarding — item performance, supplier scorecard, monthly sales trends, and pricing gap analysis.

# Supply Chain Data Model

## Medallion Architecture Overview

```
  BRONZE (raw CSV → Delta)          SILVER (cleaned/FK-validated)        GOLD (business-ready)
  ┌───────────────────────┐      ┌───────────────────────────┐      ┌─────────────────────────────┐
  │ workspace.supply_     │      │ workspace.supply_          │      │ workspace.supply_            │
  │ chain_bronze          │      │ chain_silver               │      │ chain_gold                   │
  └───────────────────────┘      └───────────────────────────┘      └─────────────────────────────┘
```

---

## Entity Relationship Diagram

```
                          ┌─────────────────────┐
                          │   s_item_master     │  DIMENSION  (100 rows)
                          │   PK: item_id       │
                          │  category, brand,   │
                          │  standard_cost, uom │
                          └──────┬───┬───┬───┬──┘
                                 │   │   │   │
              ┌──────────────────┘   │   │   └──────────────────────┐
              │                      │   │                          │
              │                      │   │                          │
  ┌───────────▼────────┐  ┌──────────▼───┐  ┌──────────▼─────────┐  ┌──▼──────────────────┐
  │ s_mrp              │  │ s_cdp_price  │  │ s_market_data_idc  │  │ s_revenue_units     │
  │ PK: item_id,       │  │ PK: item_id, │  │ PK: market_id      │  │ PK: item_id,        │
  │     region,        │  │     supplier │  │ FK: item_id        │  │     region, period  │
  │     effective_date  │  │     eff_date  │  │     region         │  │ FK: item_id         │
  │ FK: item_id        │  │ FK: item+sup │  │ (900 rows)         │  │ (3,600 rows)        │
  │ (900 rows)         │  │ (460 rows)   │  └───────────────────┘  └─────────────────────┘
  └────────────────────┘  └──────┬──────┘
                                 │
  ┌─────────────────────┐       │       ┌──────────────────────┐  ┌───────────────────────┐
  │ s_supplier_master   │◄──────┤       │ s_margin_units       │  │ s_financial_txn       │
  │ PK: supplier_id     │  bridge│       │ PK: item_id,        │  │ PK: transaction_id    │
  │ supplier_name,      │       │       │     region, period  │  │ FK: item_id          │
  │ supplier_region,    │  ┌────▼──────┐ │ FK: item_id         │  │ FK: supplier_id      │
  │ supplier_type,      │  │s_supplier│ │ (3,600 rows)        │  │ + margin_amount      │
  │ capacity, quality   │  │_terms    │ └──────────────────────┘  │ + margin_pct (calc)  │
  │ (20 rows)            │  │PK:sup+itm│                          │ (2,500 rows)          │
  └─────────────────────┘  │(460 rows)│                          └───────────────────────┘
                            └──────────┘
```

---

## Table Details

### Dimensions (Silver Layer)

| Table | Grain | PK | Key Columns | Source |
|-------|-------|----|-------------|--------|
| `s_item_master` | 1 row per item | `item_id` | item_description, category, sub_category, brand, uom, standard_cost | Item_master/item_master.csv |
| `s_supplier_master` | 1 row per supplier | `supplier_id` | supplier_name, supplier_region, supplier_type, capacity_units, lead_time_days, quality_score | supplier/supplier_master.csv |

### Bridge Table (Silver Layer)

| Table | Grain | PK | FK | Key Columns |
|-------|-------|----|----|-------------|
| `s_supplier_terms` | 1 row per supplier-item pair | `(supplier_id, item_id)` | `item_id` → s_item_master, `supplier_id` → s_supplier_master | allocated_capacity_units, minimum_order_units, target_rebate_pct, contract_status |

### Fact Tables (Silver Layer)

| Table | Grain | PK | FK | Key Columns |
|-------|-------|----|----|-------------|
| `s_financial_transactions` | 1 row per transaction | `transaction_id` | `item_id`, `supplier_id` | transaction_date, region, quantity, revenue, cost, rebate_pct, margin_amount, margin_pct |
| `s_mrp` | 1 row per item-region-date | `(item_id, region, effective_date)` | `item_id` | mrp, currency |
| `s_cdp_price` | 1 row per item-supplier-date | `(item_id, supplier_id, effective_date)` | `item_id`, `supplier_id` | cdp_price, currency |
| `s_market_data_idc` | 1 row per market entry | `market_id` | `item_id` | region, market_demand, market_price, market_share, market_date |
| `s_revenue_units` | 1 row per item-region-month | `(item_id, region, period)` | `item_id` | revenue_units |
| `s_margin_units` | 1 row per item-region-month | `(item_id, region, period)` | `item_id` | margin_units |

### Gold Layer (Business-Ready Aggregations)

| Table | Grain | Source Tables | Key Metrics |
|-------|-------|--------------|-------------|
| `g_item_performance` | 1 row per item (100) | s_item_master + s_financial_transactions + s_revenue_units + s_margin_units + s_cdp_price | total_revenue, total_cost, total_margin_amount, avg_margin_pct, total_revenue_units, total_margin_units, avg_cdp_price |
| `g_supplier_performance` | 1 row per supplier (20) | s_supplier_master + s_financial_transactions + s_supplier_terms | total_revenue, total_cost, total_margin_amount, avg_rebate_pct, contracted_items, active_contracts, capacity_utilization_pct |
| `g_monthly_sales_summary` | 1 row per month-region-category (972) | s_revenue_units + s_margin_units + s_financial_transactions + s_item_master | total_revenue_units, total_margin_units, margin_units_pct, total_revenue_amount, total_cost_amount, total_margin_amount |
| `g_pricing_analysis` | 1 row per item-region with latest MRP (363) | s_mrp + s_market_data_idc + s_cdp_price | mrp, market_price, avg_cdp_price, mrp_cdp_gap, mrp_cdp_gap_pct, mrp_market_gap, mrp_market_gap_pct |

---

## Key Relationships

```
s_item_master (1) ─── (N) s_supplier_terms (N) ─── (1) s_supplier_master
      │                                              │
      │ (1)──(N)                                     │ (1)──(N)
      ├── s_financial_transactions ◄─── FK ──┘
      ├── s_mrp
      ├── s_cdp_price ◄── FK ── s_supplier_master
      ├── s_market_data_idc
      ├── s_revenue_units
      └── s_margin_units

Gold layer derives from:
  g_item_performance      = s_item_master ⋈ s_financial_transactions ⋈ s_revenue_units ⋈ s_margin_units ⋈ s_cdp_price
  g_supplier_performance  = s_supplier_master ⋈ s_financial_transactions ⋈ s_supplier_terms
  g_monthly_sales_summary  = s_revenue_units ⋈ s_margin_units ⋈ s_financial_transactions ⋈ s_item_master
  g_pricing_analysis       = s_mrp ⋈ s_market_data_idc ⋈ s_cdp_price
```

## Foreign Key Constraints (enforced in Silver layer via subquery filters)

| From (Fact) | Column | To (Dimension) |
|-------------|--------|----------------|
| s_supplier_terms | item_id | s_item_master.item_id |
| s_supplier_terms | supplier_id | s_supplier_master.supplier_id |
| s_mrp | item_id | s_item_master.item_id |
| s_cdp_price | item_id | s_item_master.item_id |
| s_cdp_price | supplier_id | s_supplier_master.supplier_id |
| s_market_data_idc | item_id | s_item_master.item_id |
| s_revenue_units | item_id | s_item_master.item_id |
| s_margin_units | item_id | s_item_master.item_id |
| s_financial_transactions | item_id | s_item_master.item_id |
| s_financial_transactions | supplier_id | s_supplier_master.supplier_id |

In [0]:
# ---------------------------------------------------------------------------
# Configuration: create silver and gold schemas
# ---------------------------------------------------------------------------
BRONZE = "workspace.supply_chain_bronze"
SILVER = "workspace.supply_chain_silver"
GOLD = "workspace.supply_chain_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Silver schema: {SILVER}")
print(f"Gold schema:   {GOLD}")

In [0]:
# ---------------------------------------------------------------------------
# SILVER LAYER: Cleaned, deduplicated, referentially validated tables
# Build dimensions first, then facts with FK checks.
# ---------------------------------------------------------------------------

silver_statements = [
    # --- Dimensions (build first for FK references) ---
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_item_master AS
    SELECT
        trim(item_id)               AS item_id,
        trim(item_description)       AS item_description,
        trim(category)               AS category,
        trim(sub_category)           AS sub_category,
        trim(brand)                  AS brand,
        trim(uom)                    AS uom,
        standard_cost,
        current_timestamp()          AS _ingested_at
    FROM workspace.supply_chain_bronze.item_master
    WHERE item_id IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY item_id ORDER BY standard_cost DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_supplier_master AS
    SELECT
        trim(supplier_id)            AS supplier_id,
        trim(supplier_name)          AS supplier_name,
        trim(supplier_region)        AS supplier_region,
        trim(supplier_type)          AS supplier_type,
        capacity_units,
        lead_time_days,
        quality_score,
        current_timestamp()          AS _ingested_at
    FROM workspace.supply_chain_bronze.supplier_master
    WHERE supplier_id IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY supplier_id ORDER BY quality_score DESC) = 1
    """,
    # --- Fact / Bridge tables (with FK enforcement) ---
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_supplier_terms AS
    SELECT
        trim(st.supplier_id)            AS supplier_id,
        trim(st.item_id)                AS item_id,
        st.allocated_capacity_units,
        st.minimum_order_units,
        st.target_rebate_pct,
        trim(st.contract_status)        AS contract_status,
        current_timestamp()            AS _ingested_at
    FROM workspace.supply_chain_bronze.supplier_terms st
    WHERE st.supplier_id IS NOT NULL
      AND st.item_id IS NOT NULL
      AND st.supplier_id IN (SELECT supplier_id FROM workspace.supply_chain_silver.s_supplier_master)
      AND st.item_id   IN (SELECT item_id   FROM workspace.supply_chain_silver.s_item_master)
    QUALIFY row_number() OVER (PARTITION BY st.supplier_id, st.item_id ORDER BY st.target_rebate_pct DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_mrp AS
    SELECT
        trim(m.item_id)             AS item_id,
        m.mrp,
        m.effective_date,
        trim(m.currency)            AS currency,
        trim(m.region)              AS region,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.mrp m
    WHERE m.item_id IS NOT NULL
      AND m.mrp IS NOT NULL
      AND m.effective_date IS NOT NULL
      AND m.item_id IN (SELECT item_id FROM workspace.supply_chain_silver.s_item_master)
    QUALIFY row_number() OVER (PARTITION BY m.item_id, m.region, m.effective_date ORDER BY m.mrp DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_cdp_price AS
    SELECT
        trim(c.item_id)             AS item_id,
        trim(c.supplier_id)        AS supplier_id,
        c.cdp_price,
        c.effective_date,
        trim(c.currency)            AS currency,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.cdp_price c
    WHERE c.item_id IS NOT NULL
      AND c.supplier_id IS NOT NULL
      AND c.cdp_price IS NOT NULL
      AND c.effective_date IS NOT NULL
      AND c.item_id    IN (SELECT item_id    FROM workspace.supply_chain_silver.s_item_master)
      AND c.supplier_id IN (SELECT supplier_id FROM workspace.supply_chain_silver.s_supplier_master)
    QUALIFY row_number() OVER (PARTITION BY c.item_id, c.supplier_id, c.effective_date ORDER BY c.cdp_price DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_market_data_idc AS
    SELECT
        trim(mk.market_id)          AS market_id,
        trim(mk.item_id)            AS item_id,
        trim(mk.region)             AS region,
        mk.market_demand,
        mk.market_price,
        mk.market_share,
        mk.market_date,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.market_data_idc mk
    WHERE mk.market_id IS NOT NULL
      AND mk.item_id IS NOT NULL
      AND mk.item_id IN (SELECT item_id FROM workspace.supply_chain_silver.s_item_master)
    QUALIFY row_number() OVER (PARTITION BY mk.market_id ORDER BY mk.market_date DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_revenue_units AS
    SELECT
        trim(r.item_id)             AS item_id,
        trim(r.region)              AS region,
        r.revenue_units,
        r.period,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.revenue_units r
    WHERE r.item_id IS NOT NULL
      AND r.region IS NOT NULL
      AND r.period IS NOT NULL
      AND r.item_id IN (SELECT item_id FROM workspace.supply_chain_silver.s_item_master)
    QUALIFY row_number() OVER (PARTITION BY r.item_id, r.region, r.period ORDER BY r.revenue_units DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_margin_units AS
    SELECT
        trim(mg.item_id)           AS item_id,
        trim(mg.region)            AS region,
        mg.margin_units,
        mg.period,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.margin_units mg
    WHERE mg.item_id IS NOT NULL
      AND mg.region IS NOT NULL
      AND mg.period IS NOT NULL
      AND mg.item_id IN (SELECT item_id FROM workspace.supply_chain_silver.s_item_master)
    QUALIFY row_number() OVER (PARTITION BY mg.item_id, mg.region, mg.period ORDER BY mg.margin_units DESC) = 1
    """,
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_silver.s_financial_transactions AS
    SELECT
        trim(ft.transaction_id)     AS transaction_id,
        ft.transaction_date,
        trim(ft.supplier_id)        AS supplier_id,
        trim(ft.item_id)            AS item_id,
        trim(ft.region)             AS region,
        ft.quantity,
        ft.revenue,
        ft.cost,
        ft.rebate_pct,
        trim(ft.currency)           AS currency,
        (ft.revenue - ft.cost)                                          AS margin_amount,
        ((ft.revenue - ft.cost) / NULLIF(ft.revenue, 0) * 100)           AS margin_pct,
        current_timestamp()         AS _ingested_at
    FROM workspace.supply_chain_bronze.financial_transactions ft
    WHERE ft.transaction_id IS NOT NULL
      AND ft.item_id IS NOT NULL
      AND ft.supplier_id IS NOT NULL
      AND ft.item_id    IN (SELECT item_id    FROM workspace.supply_chain_silver.s_item_master)
      AND ft.supplier_id IN (SELECT supplier_id FROM workspace.supply_chain_silver.s_supplier_master)
    QUALIFY row_number() OVER (PARTITION BY ft.transaction_id ORDER BY ft.transaction_date DESC) = 1
    """,
]

# Execute each statement in order (dimensions must be built before facts)
for i, stmt in enumerate(silver_statements, 1):
    table_name = stmt.strip().split("TABLE ")[1].split(" AS")[0]
    print(f"[{i}/{len(silver_statements)}] Building {table_name} ...")
    spark.sql(stmt)
    cnt = spark.sql(f"SELECT count(*) FROM {table_name}").collect()[0][0]
    print(f"        Rows: {cnt:,}")

# Add table comments
silver_comments = {
    "s_item_master":           "Silver dimension: deduplicated item master with cleaned attributes.",
    "s_supplier_master":       "Silver dimension: deduplicated supplier master with cleaned attributes.",
    "s_supplier_terms":        "Silver bridge: supplier-item contract terms with FK validation to dimensions.",
    "s_mrp":                   "Silver fact: MRP per item-region-date, deduplicated and FK-validated.",
    "s_cdp_price":              "Silver fact: CDP price per item-supplier-date, deduplicated and FK-validated.",
    "s_market_data_idc":       "Silver fact: market demand/price/share per item-region, deduplicated and FK-validated.",
    "s_revenue_units":         "Silver fact: monthly revenue units per item-region, deduplicated and FK-validated.",
    "s_margin_units":          "Silver fact: monthly margin units per item-region, deduplicated and FK-validated.",
    "s_financial_transactions": "Silver fact: transaction-level financials with computed margin, FK-validated to dimensions.",
}
for tbl, cmt in silver_comments.items():
    spark.sql(f"ALTER TABLE {SILVER}.{tbl} SET TBLPROPERTIES (comment = '{cmt}')")

print("\nSilver layer build complete!")

In [0]:
# ---------------------------------------------------------------------------
# GOLD LAYER: Business-ready aggregated tables for analytics & dashboarding
# ---------------------------------------------------------------------------

gold_statements = [
    # 1. Item Performance — item-level KPIs enriched with master attributes
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_gold.g_item_performance AS
    WITH ft_agg AS (
        SELECT
            item_id,
            COUNT(*)                          AS transaction_count,
            SUM(quantity)                     AS total_quantity,
            SUM(revenue)                      AS total_revenue,
            SUM(cost)                         AS total_cost,
            SUM(margin_amount)                AS total_margin_amount,
            AVG(margin_pct)                   AS avg_margin_pct,
            AVG(rebate_pct)                   AS avg_rebate_pct,
            COUNT(DISTINCT supplier_id)        AS supplier_count
        FROM workspace.supply_chain_silver.s_financial_transactions
        GROUP BY item_id
    ),
    ru_agg AS (
        SELECT
            item_id,
            SUM(revenue_units)  AS total_revenue_units,
            COUNT(DISTINCT region) AS revenue_region_count
        FROM workspace.supply_chain_silver.s_revenue_units
        GROUP BY item_id
    ),
    mu_agg AS (
        SELECT
            item_id,
            SUM(margin_units) AS total_margin_units
        FROM workspace.supply_chain_silver.s_margin_units
        GROUP BY item_id
    ),
    cdp_agg AS (
        SELECT
            item_id,
            AVG(cdp_price)           AS avg_cdp_price,
            MIN(cdp_price)           AS min_cdp_price,
            MAX(cdp_price)           AS max_cdp_price,
            COUNT(DISTINCT supplier_id) AS cdp_supplier_count
        FROM workspace.supply_chain_silver.s_cdp_price
        GROUP BY item_id
    )
    SELECT
        im.item_id,
        im.item_description,
        im.category,
        im.sub_category,
        im.brand,
        im.uom,
        im.standard_cost,
        ft.transaction_count,
        ft.total_quantity,
        ft.total_revenue,
        ft.total_cost,
        ft.total_margin_amount,
        ft.avg_margin_pct,
        ft.avg_rebate_pct,
        ft.supplier_count,
        ru.total_revenue_units,
        ru.revenue_region_count,
        mu.total_margin_units,
        cdp.avg_cdp_price,
        cdp.min_cdp_price,
        cdp.max_cdp_price,
        cdp.cdp_supplier_count,
        (cdp.avg_cdp_price / NULLIF(im.standard_cost, 0) * 100) AS cdp_vs_std_cost_pct
    FROM workspace.supply_chain_silver.s_item_master im
    LEFT JOIN ft_agg  ft ON im.item_id = ft.item_id
    LEFT JOIN ru_agg  ru ON im.item_id = ru.item_id
    LEFT JOIN mu_agg  mu ON im.item_id = mu.item_id
    LEFT JOIN cdp_agg cdp ON im.item_id = cdp.item_id
    """,
    # 2. Supplier Performance — supplier scorecard with financials & contract metrics
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_gold.g_supplier_performance AS
    WITH ft_agg AS (
        SELECT
            supplier_id,
            COUNT(*)                   AS transaction_count,
            SUM(quantity)              AS total_quantity,
            SUM(revenue)               AS total_revenue,
            SUM(cost)                  AS total_cost,
            SUM(margin_amount)         AS total_margin_amount,
            AVG(margin_pct)            AS avg_margin_pct,
            AVG(rebate_pct)            AS avg_rebate_pct
        FROM workspace.supply_chain_silver.s_financial_transactions
        GROUP BY supplier_id
    ),
    st_agg AS (
        SELECT
            supplier_id,
            COUNT(*)                                       AS contracted_items,
            SUM(allocated_capacity_units)                   AS total_allocated_capacity,
            AVG(target_rebate_pct)                          AS avg_target_rebate_pct,
            SUM(CASE WHEN contract_status = 'Active' THEN 1 ELSE 0 END) AS active_contracts,
            SUM(CASE WHEN contract_status = 'Review' THEN 1 ELSE 0 END)  AS review_contracts
        FROM workspace.supply_chain_silver.s_supplier_terms
        GROUP BY supplier_id
    )
    SELECT
        sm.supplier_id,
        sm.supplier_name,
        sm.supplier_region,
        sm.supplier_type,
        sm.capacity_units,
        sm.lead_time_days,
        sm.quality_score,
        ft.transaction_count,
        ft.total_quantity,
        ft.total_revenue,
        ft.total_cost,
        ft.total_margin_amount,
        ft.avg_margin_pct,
        ft.avg_rebate_pct,
        st.contracted_items,
        st.total_allocated_capacity,
        st.active_contracts,
        st.review_contracts,
        st.avg_target_rebate_pct,
        (st.total_allocated_capacity / NULLIF(sm.capacity_units, 0) * 100) AS capacity_utilization_pct
    FROM workspace.supply_chain_silver.s_supplier_master sm
    LEFT JOIN ft_agg ft ON sm.supplier_id = ft.supplier_id
    LEFT JOIN st_agg st ON sm.supplier_id = st.supplier_id
    """,
    # 3. Monthly Sales Summary — trends by region & category (units + financials)
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_gold.g_monthly_sales_summary AS
    WITH monthly_units AS (
        SELECT
            ru.period,
            ru.region,
            im.category,
            im.sub_category,
            SUM(ru.revenue_units)  AS total_revenue_units,
            SUM(mu.margin_units)   AS total_margin_units,
            (SUM(mu.margin_units) / NULLIF(SUM(ru.revenue_units), 0) * 100) AS margin_units_pct
        FROM workspace.supply_chain_silver.s_revenue_units ru
        LEFT JOIN workspace.supply_chain_silver.s_margin_units mu
            ON ru.item_id = mu.item_id AND ru.region = mu.region AND ru.period = mu.period
        LEFT JOIN workspace.supply_chain_silver.s_item_master im
            ON ru.item_id = im.item_id
        GROUP BY ru.period, ru.region, im.category, im.sub_category
    ),
    monthly_financials AS (
        SELECT
            DATE_TRUNC('month', ft.transaction_date) AS period,
            ft.region,
            im.category,
            im.sub_category,
            SUM(ft.revenue)        AS total_revenue_amount,
            SUM(ft.cost)          AS total_cost_amount,
            SUM(ft.margin_amount)  AS total_margin_amount,
            COUNT(*)               AS transaction_count,
            SUM(ft.quantity)       AS total_quantity
        FROM workspace.supply_chain_silver.s_financial_transactions ft
        LEFT JOIN workspace.supply_chain_silver.s_item_master im
            ON ft.item_id = im.item_id
        GROUP BY DATE_TRUNC('month', ft.transaction_date), ft.region, im.category, im.sub_category
    )
    SELECT
        COALESCE(mu.period,    mf.period)         AS period,
        COALESCE(mu.region,    mf.region)          AS region,
        COALESCE(mu.category,  mf.category)       AS category,
        COALESCE(mu.sub_category, mf.sub_category)  AS sub_category,
        mu.total_revenue_units,
        mu.total_margin_units,
        mu.margin_units_pct,
        mf.total_revenue_amount,
        mf.total_cost_amount,
        mf.total_margin_amount,
        mf.transaction_count,
        mf.total_quantity
    FROM monthly_units mu
    FULL OUTER JOIN monthly_financials mf
        ON mu.period = mf.period
       AND mu.region = mf.region
       AND mu.category = mf.category
       AND mu.sub_category = mf.sub_category
    ORDER BY period, region, category, sub_category
    """,
    # 4. Pricing Analysis — MRP vs CDP vs market price gap analysis
    """
    CREATE OR REPLACE TABLE workspace.supply_chain_gold.g_pricing_analysis AS
    WITH latest_mrp AS (
        SELECT item_id, region, mrp AS latest_mrp
        FROM workspace.supply_chain_silver.s_mrp
        QUALIFY row_number() OVER (PARTITION BY item_id, region ORDER BY effective_date DESC) = 1
    ),
    avg_cdp AS (
        SELECT
            item_id,
            AVG(cdp_price)  AS avg_cdp_price,
            MIN(cdp_price)  AS min_cdp_price,
            MAX(cdp_price)  AS max_cdp_price
        FROM workspace.supply_chain_silver.s_cdp_price
        GROUP BY item_id
    ),
    market AS (
        SELECT item_id, region, market_demand, market_price, market_share
        FROM workspace.supply_chain_silver.s_market_data_idc
    )
    SELECT
        lm.item_id,
        lm.region,
        lm.latest_mrp                        AS mrp,
        m.market_price,
        m.market_demand,
        m.market_share,
        ac.avg_cdp_price,
        ac.min_cdp_price,
        ac.max_cdp_price,
        (lm.latest_mrp - ac.avg_cdp_price)                              AS mrp_cdp_gap,
        ((lm.latest_mrp - ac.avg_cdp_price) / NULLIF(lm.latest_mrp, 0) * 100) AS mrp_cdp_gap_pct,
        (lm.latest_mrp - m.market_price)                                AS mrp_market_gap,
        ((lm.latest_mrp - m.market_price) / NULLIF(lm.latest_mrp, 0) * 100)   AS mrp_market_gap_pct,
        (m.market_price - ac.avg_cdp_price)                              AS market_cdp_gap,
        ((m.market_price - ac.avg_cdp_price) / NULLIF(m.market_price, 0) * 100) AS market_cdp_gap_pct
    FROM latest_mrp lm
    LEFT JOIN market m   ON lm.item_id = m.item_id AND lm.region = m.region
    LEFT JOIN avg_cdp ac ON lm.item_id = ac.item_id
    """,
]

gold_comments = {
    "g_item_performance":       "Gold: item-level performance scorecard with financial KPIs, unit volumes, and pricing from CDP.",
    "g_supplier_performance":   "Gold: supplier scorecard with transaction metrics, contract status, and capacity utilization.",
    "g_monthly_sales_summary":  "Gold: monthly sales trends by region and category — revenue units, margin units, and financial amounts.",
    "g_pricing_analysis":       "Gold: pricing gap analysis comparing MRP, CDP cost price, and market price per item-region.",
}

for i, stmt in enumerate(gold_statements, 1):
    table_name = stmt.strip().split("TABLE ")[1].split(" AS")[0]
    print(f"[{i}/{len(gold_statements)}] Building {table_name} ...")
    spark.sql(stmt)
    cnt = spark.sql(f"SELECT count(*) FROM {table_name}").collect()[0][0]
    print(f"        Rows: {cnt:,}")

for tbl, cmt in gold_comments.items():
    spark.sql(f"ALTER TABLE {GOLD}.{tbl} SET TBLPROPERTIES (comment = '{cmt}')")

print("\nGold layer build complete!")

In [0]:
# ---------------------------------------------------------------------------
# Validation: row counts for all silver and gold tables
# ---------------------------------------------------------------------------

silver_tables = [
    "s_item_master", "s_supplier_master", "s_supplier_terms",
    "s_mrp", "s_cdp_price", "s_market_data_idc",
    "s_revenue_units", "s_margin_units", "s_financial_transactions",
]
gold_tables = [
    "g_item_performance", "g_supplier_performance",
    "g_monthly_sales_summary", "g_pricing_analysis",
]

print("SILVER LAYER")
print(f"{'Table':<30} {'Rows':>10}")
print("-" * 42)
for t in silver_tables:
    cnt = spark.sql(f"SELECT count(*) FROM {SILVER}.{t}").collect()[0][0]
    print(f"{t:<30} {cnt:>10,}")

print(f"\nGOLD LAYER")
print(f"{'Table':<30} {'Rows':>10}")
print("-" * 42)
for t in gold_tables:
    cnt = spark.sql(f"SELECT count(*) FROM {GOLD}.{t}").collect()[0][0]
    print(f"{t:<30} {cnt:>10,}")

# Show samples from gold tables
print("\n" + "=" * 50)
print("GOLD TABLE SAMPLES")
print("=" * 50)

for t in gold_tables:
    print(f"\n--- {GOLD}.{t} ---")
    display(spark.sql(f"SELECT * FROM {GOLD}.{t} LIMIT 5"))

In [0]:
# ---------------------------------------------------------------------------
# Data Quality Checks on Silver Layer
# Categories: Row Count, Duplicates, Nulls, FK Integrity, Value Range, Domain
# ---------------------------------------------------------------------------
SILVER = "workspace.supply_chain_silver"
BRONZE = "workspace.supply_chain_bronze"

checks = []  # (category, table, description, failure_count, status)

# === 1. Row Count Comparison (Bronze → Silver) ===
rc_pairs = [
    ("item_master", "s_item_master"),
    ("supplier_master", "s_supplier_master"),
    ("supplier_terms", "s_supplier_terms"),
    ("mrp", "s_mrp"),
    ("cdp_price", "s_cdp_price"),
    ("market_data_idc", "s_market_data_idc"),
    ("revenue_units", "s_revenue_units"),
    ("margin_units", "s_margin_units"),
    ("financial_transactions", "s_financial_transactions"),
]
for b, s in rc_pairs:
    bc = spark.sql(f"SELECT count(*) FROM {BRONZE}.{b}").collect()[0][0]
    sc = spark.sql(f"SELECT count(*) FROM {SILVER}.{s}").collect()[0][0]
    dropped = bc - sc
    checks.append(("Row Count", s, f"{bc:,} -> {sc:,}", int(dropped),
                   "PASS" if dropped == 0 else "WARN"))

# === 2. Duplicate Detection on Primary Keys ===
dup_checks = [
    ("s_item_master", "item_id"),
    ("s_supplier_master", "supplier_id"),
    ("s_supplier_terms", "supplier_id, item_id"),
    ("s_mrp", "item_id, region, effective_date"),
    ("s_cdp_price", "item_id, supplier_id, effective_date"),
    ("s_market_data_idc", "market_id"),
    ("s_revenue_units", "item_id, region, period"),
    ("s_margin_units", "item_id, region, period"),
    ("s_financial_transactions", "transaction_id"),
]
for table, pk in dup_checks:
    dups = spark.sql(f"""
        SELECT count(*) FROM (
            SELECT {pk}, count(*) AS cnt
            FROM {SILVER}.{table}
            GROUP BY {pk}
            HAVING count(*) > 1
        )
    """).collect()[0][0]
    checks.append(("Duplicates", table, f"PK: ({pk})", int(dups),
                   "PASS" if dups == 0 else "FAIL"))

# === 3. Null Checks on Critical Columns ===
null_checks = [
    ("s_item_master", "item_id IS NULL"),
    ("s_item_master", "category IS NULL"),
    ("s_item_master", "standard_cost IS NULL"),
    ("s_supplier_master", "supplier_id IS NULL"),
    ("s_supplier_master", "supplier_name IS NULL"),
    ("s_supplier_terms", "supplier_id IS NULL OR item_id IS NULL"),
    ("s_mrp", "item_id IS NULL OR mrp IS NULL OR effective_date IS NULL"),
    ("s_cdp_price", "item_id IS NULL OR supplier_id IS NULL OR cdp_price IS NULL"),
    ("s_market_data_idc", "market_id IS NULL OR item_id IS NULL"),
    ("s_revenue_units", "item_id IS NULL OR region IS NULL OR period IS NULL OR revenue_units IS NULL"),
    ("s_margin_units", "item_id IS NULL OR region IS NULL OR period IS NULL OR margin_units IS NULL"),
    ("s_financial_transactions", "transaction_id IS NULL OR item_id IS NULL OR supplier_id IS NULL"),
    ("s_financial_transactions", "revenue IS NULL OR cost IS NULL OR quantity IS NULL"),
]
for table, cond in null_checks:
    nulls = spark.sql(f"SELECT count(*) FROM {SILVER}.{table} WHERE {cond}").collect()[0][0]
    checks.append(("Nulls", table, cond, int(nulls),
                   "PASS" if nulls == 0 else "FAIL"))

# === 4. Referential Integrity (FK) Checks ===
fk_checks = [
    ("s_supplier_terms", "supplier_id", f"supplier_id NOT IN (SELECT supplier_id FROM {SILVER}.s_supplier_master)"),
    ("s_supplier_terms", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_mrp", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_cdp_price", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_cdp_price", "supplier_id", f"supplier_id NOT IN (SELECT supplier_id FROM {SILVER}.s_supplier_master)"),
    ("s_market_data_idc", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_revenue_units", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_margin_units", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_financial_transactions", "item_id", f"item_id NOT IN (SELECT item_id FROM {SILVER}.s_item_master)"),
    ("s_financial_transactions", "supplier_id", f"supplier_id NOT IN (SELECT supplier_id FROM {SILVER}.s_supplier_master)"),
]
for table, col, cond in fk_checks:
    orphans = spark.sql(f"SELECT count(*) FROM {SILVER}.{table} WHERE {cond}").collect()[0][0]
    checks.append(("FK Integrity", table, f"{col} orphan check", int(orphans),
                   "PASS" if orphans == 0 else "FAIL"))

# === 5. Value Range / Boundary Checks ===
range_checks = [
    ("s_item_master", "standard_cost <= 0", "standard_cost must be > 0"),
    ("s_supplier_master", "quality_score < 0 OR quality_score > 1", "quality_score must be 0-1"),
    ("s_supplier_master", "capacity_units <= 0", "capacity_units must be > 0"),
    ("s_supplier_master", "lead_time_days <= 0", "lead_time_days must be > 0"),
    ("s_supplier_terms", "allocated_capacity_units <= 0", "allocated_capacity must be > 0"),
    ("s_supplier_terms", "target_rebate_pct < 0 OR target_rebate_pct > 100", "rebate_pct must be 0-100"),
    ("s_mrp", "mrp <= 0", "mrp must be > 0"),
    ("s_cdp_price", "cdp_price <= 0", "cdp_price must be > 0"),
    ("s_market_data_idc", "market_share < 0 OR market_share > 1", "market_share must be 0-1"),
    ("s_market_data_idc", "market_price <= 0", "market_price must be > 0"),
    ("s_market_data_idc", "market_demand <= 0", "market_demand must be > 0"),
    ("s_revenue_units", "revenue_units < 0", "revenue_units must be >= 0"),
    ("s_margin_units", "margin_units < 0", "margin_units must be >= 0"),
    ("s_financial_transactions", "revenue <= 0", "revenue must be > 0"),
    ("s_financial_transactions", "cost <= 0", "cost must be > 0"),
    ("s_financial_transactions", "quantity <= 0", "quantity must be > 0"),
    ("s_financial_transactions", "margin_amount <> (revenue - cost)", "margin_amount must equal revenue - cost"),
]
for table, cond, desc in range_checks:
    viol = spark.sql(f"SELECT count(*) FROM {SILVER}.{table} WHERE {cond}").collect()[0][0]
    checks.append(("Value Range", table, desc, int(viol),
                   "PASS" if viol == 0 else "FAIL"))

# === 6. Domain / Enum Checks ===
expected_regions = "('North America', 'Europe', 'Asia Pacific', 'Latin America')"
domain_checks = [
    ("s_supplier_master", f"supplier_type NOT IN ('Strategic', 'Preferred', 'Standard')", "supplier_type enum"),
    ("s_supplier_master", f"supplier_region NOT IN {expected_regions}", "supplier_region enum"),
    ("s_supplier_terms", "contract_status NOT IN ('Active', 'Review')", "contract_status enum"),
    ("s_mrp", f"region NOT IN {expected_regions}", "region enum (mrp)"),
    ("s_market_data_idc", f"region NOT IN {expected_regions}", "region enum (market)"),
    ("s_revenue_units", f"region NOT IN {expected_regions}", "region enum (revenue)"),
    ("s_margin_units", f"region NOT IN {expected_regions}", "region enum (margin)"),
    ("s_financial_transactions", f"region NOT IN {expected_regions}", "region enum (fin_txns)"),
    ("s_mrp", "currency <> 'USD'", "currency must be USD (mrp)"),
    ("s_cdp_price", "currency <> 'USD'", "currency must be USD (cdp)"),
    ("s_financial_transactions", "currency <> 'USD'", "currency must be USD (fin_txns)"),
]
for table, cond, desc in domain_checks:
    viol = spark.sql(f"SELECT count(*) FROM {SILVER}.{table} WHERE {cond}").collect()[0][0]
    checks.append(("Domain", table, desc, int(viol),
                   "PASS" if viol == 0 else "FAIL"))

# === Build Summary DataFrame ===
from pyspark.sql.types import StructType, StructField, StringType, LongType

dq_schema = StructType([
    StructField("category", StringType(), True),
    StructField("table", StringType(), True),
    StructField("check", StringType(), True),
    StructField("failures", LongType(), True),
    StructField("status", StringType(), True),
])
dq_df = spark.createDataFrame(checks, dq_schema)
print("DATA QUALITY CHECK RESULTS")
print("=" * 80)
display(dq_df.orderBy("category", "table"))

total = len(checks)
passed = sum(1 for c in checks if c[4] == "PASS")
warned = sum(1 for c in checks if c[4] == "WARN")
failed = sum(1 for c in checks if c[4] == "FAIL")
print(f"\nSummary: {passed} PASS  |  {warned} WARN  |  {failed} FAIL  |  Total: {total} checks")
if failed > 0:
    print("\n*** FAILURES DETECTED — review the DQ Failure Details cell below ***")
else:
    print("\nAll critical checks passed.")

In [0]:
# ---------------------------------------------------------------------------
# DQ Failure Details: show specific rows that violate any failed check
# Only displays output if failures were found above.
# ---------------------------------------------------------------------------

failed_checks = [c for c in checks if c[4] == "FAIL"]

if not failed_checks:
    print("No FAIL-level violations found. Showing WARN details if any:")
    warn_checks = [c for c in checks if c[4] == "WARN"]
    if not warn_checks:
        print("  No warnings either — all silver tables are clean.")
    else:
        for cat, tbl, desc, cnt, status in warn_checks:
            print(f"  [{status}] {tbl}: {desc}  ({cnt} rows affected)")
            if "->" in desc:  # Row count comparison
                b_name = tbl.replace("s_", "")
                bc = spark.sql(f"SELECT count(*) FROM {BRONZE}.{b_name}").collect()[0][0]
                sc = spark.sql(f"SELECT count(*) FROM {SILVER}.{tbl}").collect()[0][0]
                print(f"       Bronze: {bc:,}  Silver: {sc:,}  Dropped: {bc - sc:,}")
else:
    print(f"{len(failed_checks)} check(s) FAILED. Showing violating rows:\n")
    for cat, tbl, desc, cnt, status in failed_checks:
        print(f"\n{'='*60}")
        print(f"FAIL: {tbl} — {desc}  ({cnt} violating rows)")
        print(f"{'='*60}")

        # Show up to 10 violating rows for each failed check
        if cat == "Duplicates":
            pk = desc.replace("PK: ", "").strip("()")
            display(spark.sql(f"""
                SELECT {pk}, count(*) AS dup_count
                FROM {SILVER}.{tbl}
                GROUP BY {pk}
                HAVING count(*) > 1
                ORDER BY dup_count DESC
                LIMIT 10
            """))
        elif cat == "Nulls":
            display(spark.sql(f"SELECT * FROM {SILVER}.{tbl} WHERE {desc} LIMIT 10"))
        elif cat == "FK Integrity":
            col = desc.replace(" orphan check", "")
            display(spark.sql(f"SELECT * FROM {SILVER}.{tbl} WHERE {col} NOT IN (SELECT {col} FROM {SILVER}.s_{'supplier_master' if 'supplier' in col else 'item_master'}) LIMIT 10"))
        elif cat == "Value Range":
            # Reconstruct the condition from the description
            range_map = {
                "standard_cost must be > 0": "standard_cost <= 0",
                "quality_score must be 0-1": "quality_score < 0 OR quality_score > 1",
                "capacity_units must be > 0": "capacity_units <= 0",
                "lead_time_days must be > 0": "lead_time_days <= 0",
                "allocated_capacity must be > 0": "allocated_capacity_units <= 0",
                "rebate_pct must be 0-100": "target_rebate_pct < 0 OR target_rebate_pct > 100",
                "mrp must be > 0": "mrp <= 0",
                "cdp_price must be > 0": "cdp_price <= 0",
                "market_share must be 0-1": "market_share < 0 OR market_share > 1",
                "market_price must be > 0": "market_price <= 0",
                "market_demand must be > 0": "market_demand <= 0",
                "revenue_units must be >= 0": "revenue_units < 0",
                "margin_units must be >= 0": "margin_units < 0",
                "revenue must be > 0": "revenue <= 0",
                "cost must be > 0": "cost <= 0",
                "quantity must be > 0": "quantity <= 0",
                "margin_amount must equal revenue - cost": "margin_amount <> (revenue - cost)",
            }
            cond = range_map.get(desc, "1=0")
            display(spark.sql(f"SELECT * FROM {SILVER}.{tbl} WHERE {cond} LIMIT 10"))
        elif cat == "Domain":
            domain_map = {
                "supplier_type enum": "supplier_type NOT IN ('Strategic', 'Preferred', 'Standard')",
                "supplier_region enum": "supplier_region NOT IN ('North America', 'Europe', 'Asia Pacific', 'Latin America')",
                "contract_status enum": "contract_status NOT IN ('Active', 'Review')",
            }
            cond = domain_map.get(desc, "1=0")
            if cond != "1=0":
                display(spark.sql(f"SELECT * FROM {SILVER}.{tbl} WHERE {cond} LIMIT 10"))
            else:
                print(f"  (Domain check for '{desc}' — see summary above)")